# Model 2 — Detecção de Placa dentro do Carro

Este notebook treina o **segundo modelo YOLO** do pipeline ANPR.

**Função:** Recebe uma imagem de carro (já recortado pelo Model 1) e retorna onde está a placa.

**Pipeline completo:**
```
Câmera → [Model 1: detecta carro] → [Model 2: detecta placa no carro] → [OCR: lê o texto]
```

**Dataset:** `dataset_plate_crop` — gerado pelo script `crop_dataset.py`
- Train: 292 imagens de carros recortados com label de placa
- Valid: 79 imagens
- Test: 38 imagens
- Classe: `license-plate`

In [ ]:
import os
from ultralytics import YOLO

# Modelo pré-treinado (mesmo usado pelo Gustavo)
# Troque para 'yolov8n.pt' se não tiver o yolo26n.pt disponível
model = YOLO('yolo26n.pt')

config_path = '../datasets/dataset_plate_crop/data.yaml'

print(f'Dataset: {config_path}')
print(f'Modelo base: {model.model_name}')

In [ ]:
# Treino do Model 2
results = model.train(
    data=config_path,
    epochs=200,
    imgsz=640,
    batch=32,
    patience=25,       # para cedo se não melhorar
    plots=True,
    name='model-plate'
)

In [ ]:
# Avaliação no conjunto de validação
metrics = model.val(data=config_path)

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")

In [ ]:
# Localização do melhor modelo salvo
best_model_path = results.save_dir + '/weights/best.pt'
print(f'Melhor modelo salvo em: {best_model_path}')